# Part 1b: Getting started with PyTorch

In this notebook we train the same network architecture on the LHC jet tagging dataset using PyTorch. When you are done, head straight to **`1c_hls4ml_synth.ipynb`** to convert the model to an FPGA design.

In [ ]:
import numpy as np
import os
import sys

sys.path.append('..')

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

%matplotlib inline
np.random.seed(0)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)

## Fetch the jet tagging dataset from Open ML

The [HLS4ML LHC jet dataset](https://openml.org/search?type=data&id=42468) was introduced in [Duarte et al. (2018)](https://arxiv.org/abs/1804.06913) to benchmark fast neural network inference on FPGAs for particle physics applications.

Jets are collimated sprays of particles produced when quarks or gluons are knocked out of colliding protons at the LHC. Identifying the origin of a jet in real time is a core task for LHC trigger systems, which must decide within a few microseconds whether to keep or discard each collision event.

The dataset contains 16 high-level jet substructure observables derived from simulated proton-proton collisions at √s = 13 TeV. These include energy correlation functions, N-subjettiness ratios, a groomed jet mass, and constituent multiplicity. The goal is to classify each jet into one of five categories:

| Label | Jet origin |
|-------|------------|
| `g`   | Gluon |
| `q`   | Light quark |
| `w`   | W boson decay (W → qq') |
| `z`   | Z boson decay (Z → qq') |
| `t`   | Top quark decay (t → bqq') |

In [ ]:
data = fetch_openml('hls4ml_lhc_jets_hlf')
X, y = data['data'], data['target']

### Let's print some information about the dataset


In [ ]:
print(data['feature_names'])
print(X.shape, y.shape)
print(X[:5])
print(y[:5])

As you saw above, the `y` target is an array of strings, e.g. `['g', 'w', ...]` etc.
We need to make this a "One Hot" encoding for the training.
Then, split the dataset into training and validation sets:

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)
y = np.eye(5)[y_encoded]  # one-hot encode
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(y[:5])

In [ ]:
scaler = StandardScaler()
X_train_val = scaler.fit_transform(X_train_val)
X_test = scaler.transform(X_test)

os.makedirs('../data/jet-tagging', exist_ok=True)
np.save('../data/jet-tagging/X_train_val.npy', X_train_val)
np.save('../data/jet-tagging/X_test.npy', X_test)
np.save('../data/jet-tagging/y_train_val.npy', y_train_val)
np.save('../data/jet-tagging/y_test.npy', y_test)
np.save('../data/jet-tagging/classes.npy', le.classes_)

## Now construct a model
We'll use 3 hidden layers with 64, then 32, then 32 neurons with ReLU activation, and a 5-neuron output with Softmax.

Note: unlike Keras, PyTorch's `CrossEntropyLoss` fuses LogSoftmax and NLLLoss internally and therefore expects raw logits. Because Softmax is part of our model, we instead use `NLLLoss` with the log of the model output, which is equivalent.

In [ ]:
class JetTagger(nn.Module):
    """Simple 3-hidden-layer jet tagger: 16 → 64 → 32 → 32 → 5."""

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(16, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 32)
        self.output = nn.Linear(32, 5)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        return torch.softmax(self.output(x), dim=1)


model = JetTagger()
print(model)

## Train the model
We'll use the Adam optimiser with NLL loss.
The model isn't very complex, so this should take just a few minutes even on the CPU.

In [ ]:
n_train = int(len(X_train_val) * 0.75)

X_tr = torch.FloatTensor(X_train_val[:n_train])
y_tr = torch.LongTensor(np.argmax(y_train_val[:n_train], axis=1))
X_val = torch.FloatTensor(X_train_val[n_train:])
y_val = torch.LongTensor(np.argmax(y_train_val[n_train:], axis=1))

loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=1024, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.NLLLoss()

for epoch in range(20):
    model.train()
    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        loss = criterion(torch.log(model(X_batch).clamp(min=1e-7)), y_batch)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss = criterion(torch.log(model(X_val).clamp(min=1e-7)), y_val).item()
    print(f'Epoch {epoch + 1:2d}  val_loss={val_loss:.4f}')

os.makedirs('../models', exist_ok=True)
torch.save(model.state_dict(), '../models/pytorch_weights_part1.pt')
print('Saved ../models/pytorch_weights_part1.pt')

## Check performance
Check the accuracy and make a ROC curve:

In [ ]:
import plotting
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

model.eval()
with torch.no_grad():
    y_pytorch = model(torch.FloatTensor(X_test)).numpy()

print("Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_pytorch, axis=1))))
plt.figure(figsize=(9, 9))
_ = plotting.makeRoc(y_test, y_pytorch, le.classes_)

An accuracy of ~75% is expected for this 5-class problem — random guessing gives only 20%, and some classes (notably gluon vs. light quark) are physically very similar and genuinely hard to separate even with more sophisticated methods.

The ROC (Receiver Operating Characteristic) curve shows, for each class, the trade-off between signal efficiency (true positive rate) and background efficiency (false positive rate) as the decision threshold is varied. The area under the curve (AUC) ranges from 0.5 (random classifier) to 1.0 (perfect). Higher and further to the upper-left is better.

**N.B.** This notebook trains a full-precision (32-bit floating-point) model. When converting to an FPGA design, hls4ml applies post-training quantization (PTQ) by default, which works well at 16-bit precision but struggles to match accuracy below ~8 bits. For the most resource-efficient FPGA designs, **quantization-aware training (QAT)** gives substantially better results. See **Part 2** for QKeras (Keras) and Brevitas (PyTorch) QAT examples.

## Next step

Your model is trained and saved. Open **`1c_hls4ml_synth.ipynb`** to convert it to an FPGA design with hls4ml.

## Further reading

For more details, see: Duarte, Han, Harris et al., "Fast inference of deep neural networks in FPGAs for particle physics", JINST 13 P07027 (2018), [arXiv:1804.06913](https://arxiv.org/abs/1804.06913)